# Burfisher 3e — TBL65B (= ME3C): subsidio a MFG de USA con desempleo

Mismo shock que TBL65A, un subsidio del 10% a la producción de MFG en USA
(`to("MFG","MFG","USA") = -10.9414`), pero con **desempleo**: el salario real de USA
queda fijo y lo que se ajusta es el empleo.

En GEMPACK (`ME3C.EXP`) el cierre es `swap qe("LABOR","USA") = pebfactreal("LABOR","USA")`.
Aquí se arma modificando dos bloques **sólo en este notebook** con `@overwrite`.

In [ ]:
from pathlib import Path

from pyomo.environ import value

from equilibria._local_refs import nus333_dir
from equilibria.blocks.gtap import ClosureBlock, ShockBlock, overwrite, ppriv_tornqvist
from equilibria.templates.gtap import GTAPParameters
from equilibria.templates.gtap.gtap_block_model import build_block_model
from equilibria.templates.gtap.gtap_contract import GTAPClosureConfig
from equilibria.templates.gtap.gtap_multiperiod_driver import solve_multiperiod
from equilibria.templates.gtap.instruments import fix_instrument_shock

## 1. Datos: la base nus333 (2 regiones × 3 sectores) y el `.prm` del ejercicio

In [ ]:
har: Path = nus333_dir()
p = GTAPParameters()
p.load_from_har(
    basedata_path=har / "basedata.har",
    sets_path=har / "sets.har",
    default_path=har / "default.prm",
    baserate_path=har / "baserate.har",
)

## 2. El cierre de desempleo, con `@overwrite`

- **`ShockBlock`**: `aft[USA,LABOR]` (la oferta de trabajo) deja de ser un instrumento
  fijo y pasa a ser endógena.
- **`ClosureBlock`**: una fila nueva, `eq_wreal`, que fija el salario real:
  `pft = pft_base × T`, donde `T` es el índice de precios al consumidor de los hogares
  (Törnqvist contra la base).

¿Por qué Törnqvist? El `ppriv` de GEMPACK es un índice Divisia, que acumula los cambios
de precio paso a paso. El Törnqvist es ese mismo índice en un solo paso. Medido con
el oráculo GAMS: con Törnqvist el empleo da +38,781%, contra +38,783% en GEMPACK. Con
`pcons` daba +37,01%.

Con `period="shock"` el cierre cambia sólo en el shock, como el `swap` de GEMPACK:
en base y check `aft` queda fijo y rige el pleno empleo.

Los hooks quedan registrados en la clase y duran todo el notebook. Si re-ejecutas la
celda, el hook se reemplaza, no se duplica.

In [ ]:
@overwrite(ShockBlock, period="shock")
def unemployment(b):
    b.endogenous("aft", ("USA", "LABOR"))


@overwrite(ClosureBlock, period="shock")
def real_wage(b):
    # Al construir la fila, value(...) es el nivel de base (constante).
    b.equation(
        "eq_wreal",
        ("USA", "LABOR"),
        lambda m, r, f: m.pft[r, f] == value(m.pft[r, f]) * ppriv_tornqvist(m, r),
    )

## 3. Cierre global y modelo

Mismos flags que TBL65A. `build_block_model` arma los 3 períodos (base / check / shock)
con la lista de bloques `GTAP_BLOCK_ORDER`, y ya incluye los hooks de arriba.

In [ ]:
gc = GTAPClosureConfig(
    name="base",
    closure_type="MCP",
    capital_mobility="sluggish",
    fix_endowments=False,
    fix_taxes=False,
    fix_technology=False,
    if_sub=False,
    savf_flag="capFlex",
    numeraire="pnum",
)
m, _ = build_block_model(p, p.sets, gc, "ROW", base_calibrated=False, ref_gdx=None)
r, f = "USA", "LABOR"
print("aft[USA,LABOR] libre:", not m.aft[r, f, "shock"].fixed)
print("aft[ROW,LABOR] fijo: ", m.aft["ROW", f, "shock"].fixed)
print("fila eq_wreal:       ", list(m.eq_wreal))

## 4. El shock: sólo cambia el instrumento en el período `shock`

In [ ]:
idx = ("USA", "MFG", "MFG")
t0 = value(m.prdtx_rai[(*idx, "check")])
nuevo = (1.0 + t0) * (1.0 - 0.109414) - 1.0
fix_instrument_shock(m, "prdtx_rai", idx, value=nuevo)
print(f"prdtx_rai{idx}: check {t0:.6f} -> shock {nuevo:.6f}")

## 5. Resolver check y shock

In [ ]:
res = solve_multiperiod(
    m,
    p,
    gc,
    ref_gdx=None,
    skip_base_solve=True,  # GAMS no resuelve la base
    mute_welfare=True,
    seed_from_prior=False,
    mode="gtap",
    solve_check=True,
)
print("codes:", {k: int(res[k]["code"]) for k in res})

## 6. Resultados (% de check a shock) contra GAMS y GEMPACK

In [ ]:
def pct(var, *k):
    comp = getattr(m, var)
    return 100.0 * (value(comp[(*k, "shock")]) / value(comp[(*k, "check")]) - 1.0)


filas = [
    ("empleo USA (xft LABOR)", pct("xft", r, f), 38.780869, 38.7830),
    ("salario USA (pft LABOR)", pct("pft", r, f), 7.691635, 7.7126),
    ("producción AGR USA (xp)", pct("xp", "USA", "AGR"), 4.001585, 4.0016),
    ("producción MFG USA (xp)", pct("xp", "USA", "MFG"), 28.312494, 28.3127),
    ("producción SER USA (xp)", pct("xp", "USA", "SER"), 26.880396, 26.8821),
]
print(f"{'':26s}{'equilibria':>12s}{'GAMS':>12s}{'GEMPACK':>12s}")
for nombre, eq, gams, gem in filas:
    print(f"{nombre:26s}{eq:+12.4f}{gams:+12.4f}{gem:+12.4f}")

El salario nominal de GEMPACK difiere en un factor uniforme de 1,0002 en todos los
precios: es el nivel del numerario, no un efecto real. El salario **real** es 0 en los
tres.